In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys

# Clear any existing handlers to prevent duplicates
root_logger = logging.getLogger()
if root_logger.handlers:
    for handler in root_logger.handlers:
        root_logger.removeHandler(handler)

# Configure logging FIRST before importing DataLoader
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress specific loggers that are verbose
for logger_name in ['__main__', 'snowflake', 'azure', 'urllib3']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Add the project root to the path to import custom modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import the DataLoader class
from scripts.data_preparation.load import DataLoader

print("✓ Libraries imported successfully")
print("✓ Logging configured (INFO messages suppressed)")

✓ Libraries imported successfully
✓ Logging configured (INFO messages suppressed)


In [2]:
# Getting email data from startdate to enddate
startdate = '2026-05-15'
# enddate = '2026-05-27' # '2026-05-11'

print(f"Tracking Email History will be Retrieved from {startdate}")

Tracking Email History will be Retrieved from 2026-05-15


In [3]:
# Force reload the DataLoader module to ensure latest code is used
import importlib
import sys

# Remove from cache if present, then reimport fresh
if 'scripts.data_preparation.load' in sys.modules:
    del sys.modules['scripts.data_preparation.load']

from scripts.data_preparation.load import DataLoader
print("✓ DataLoader module loaded fresh")

# Verify the parameter substitution method being used
import inspect
src = inspect.getsource(DataLoader.load_from_snowflake)
if 'query.replace' in src:
    print("✓ Using str.replace() for parameter substitution (correct)")
elif 'query.format' in src:
    print("⚠️  Still using str.format() - reload did not pick up changes!")


✓ DataLoader module loaded fresh
⚠️  Still using str.format() - reload did not pick up changes!


## 1. Load Tracking CheckCall Data from Snowflake

Load the tracking checkcall data from Snowflake using the DataLoader class.

In [4]:
# Execute the query to get historical load data
print("Querying Snowflake for Tracking checkcall data...")
print(f"Date Range: {startdate} to Current Date")

loader = DataLoader()

df = loader.load_from_snowflake(
    sql_file_path='../sql/loadTrackingOverprocessing.sql',
    params={'startdate': startdate}
)

# Convert datetime column immediately after load (handles mixed types / NaN floats)
df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')

print(f"\n✓ Loaded {len(df):,} rows from Snowflake")
print(f"  Date range in data: {df['ENTERED_DATETIME_CST'].min()} to {df['ENTERED_DATETIME_CST'].max()}")
print(f"  Columns: {len(df.columns)}")
df.head()


Querying Snowflake for Tracking checkcall data...
Date Range: 2026-05-15 to Current Date


/home/azureuser/loadTrackingOverProcessing/LoadTrackingOverProcessing/scripts/data_preparation/load.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.snowflake_conn)



✓ Loaded 28,204,130 rows from Snowflake
  Date range in data: 2002-05-19 16:30:00+00:00 to 2033-06-02 14:00:00+00:00
  Columns: 22


,LOAD_NUM,CHECK_CALL_TYPE,DESCRIPTION,ENTERED_DATETIME_CST,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,AUTOMATED,...,UPDATE_USER,HUMAN_ENTERED_CHECKCALL_FLAG,TRACKING_IDENTIFIER_CLEAN,TRACKING_IDENTIFIER_TYPE,TRACKING_METHOD_TYPE,TRACKING_SLA_PRE_PICK,TRACKING_SLA_IN_TRANSIT,TRACKING_SLA_TOTAL,TRACKING_SLA_TOTAL_SCORE_ACTUAL,TRACKING_SLA_TOTAL_SCORE_POSSIBLE
0,524849714,P-Open,W63713287,2026-05-26 14:00:00+00:00,Pocomoke City,MD,United States,38.079201,-75.589500,None,...,None,NaN,None,None,None,1.0,0.0,1.0,5.0,5.0
1,524849714,DP,Depart Pick,2026-05-26 17:43:00+00:00,POCOMOKE CITY,MD,US,38.079201,-75.589500,True,...,API-GPS,0.0,4436693691,DRIVERPHONE,MACROPOINTAPP,1.0,0.0,1.0,5.0,5.0
2,524849714,CC,Check Call,2026-05-26 17:43:00+00:00,WOODBRIDGE,NJ,US,40.546004,-74.282822,True,...,MACROPOINT,0.0,4436693691,DRIVERPHONE,MACROPOINTAPP,1.0,0.0,1.0,5.0,5.0
3,524849714,AP,Arrived Pick,2026-05-26 17:43:00+00:00,POCOMOKE CITY,MD,US,38.079201,-75.589500,True,...,API-GPS,0.0,4436693691,DRIVERPHONE,MACROPOINTAPP,1.0,0.0,1.0,5.0,5.0
4,524849714,CC,Check Call,2026-05-26 17:53:00+00:00,ELIZABETH,NJ,US,40.670944,-74.176172,True,...,MACROPOINT,0.0,4436693691,DRIVERPHONE,MACROPOINTAPP,1.0,0.0,1.0,5.0,5.0


In [5]:
def get_loads_with_manual_cc_before_pickup(df):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded before
    the pickup open event (CHECK_CALL_TYPE = 'P-Open').
    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - first_pickup_open: earliest P-Open datetime
                      - manual_cc_before_pickup_count: number of manual CCs before P-Open
                      - earliest_manual_cc: datetime of the first manual CC before P-Open
    """
    # Ensure datetime column is parsed (utc=True handles tz-aware mixed values)
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True)

    # Get the first P-Open datetime per load
    pickup_open = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
        .reset_index()
    )

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Join manual CCs with their load's first P-Open datetime
    manual_cc = manual_cc.merge(pickup_open, on='LOAD_NUM', how='inner')

    # Keep only manual CCs that occurred BEFORE the first P-Open
    manual_cc_before = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] < manual_cc['first_pickup_open']
    ]

    # Aggregate per load
    result = (
        manual_cc_before.groupby(['LOAD_NUM', 'first_pickup_open'])
        .agg(
            manual_cc_before_pickup_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_manual_cc=('ENTERED_DATETIME_CST', 'min')
        )
        .reset_index()
        .sort_values('manual_cc_before_pickup_count', ascending=False)
    )

    return result


# Run the function
total_loads = df['LOAD_NUM'].nunique()
loads_with_early_manual_cc = get_loads_with_manual_cc_before_pickup(df)
qualifying_loads = len(loads_with_early_manual_cc)

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC before pickup open:       {qualifying_loads:,}")
print(f"Percentage:                                        {qualifying_loads / total_loads * 100:.1f}%")
loads_with_early_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               224,654
LOAD_NUMs with manual CC before pickup open:       7,397
Percentage:                                        3.3%


,LOAD_NUM,first_pickup_open,manual_cc_before_pickup_count,earliest_manual_cc
4962,554474874,2026-05-26 14:00:00+00:00,6,2026-05-22 00:00:00+00:00
3228,553916777,2026-05-23 07:01:00+00:00,6,2026-05-22 02:27:00+00:00
2848,553765055,2026-05-22 16:30:00+00:00,4,2026-05-22 13:34:00+00:00
2945,553782702,2026-05-18 20:00:00+00:00,4,2026-05-18 13:55:00+00:00
4944,554469157,2026-05-29 20:00:00+00:00,4,2026-05-28 15:41:00+00:00
4520,554302888,2026-06-26 21:00:00+00:00,4,2026-05-26 16:42:00+00:00
5988,555023315,2026-06-01 16:00:00+00:00,4,2026-06-01 12:01:00+00:00
4488,554294417,2026-05-21 15:00:00+00:00,4,2026-05-21 01:06:00+00:00
4963,554475186,2026-05-26 20:00:00+00:00,4,2026-05-26 15:09:00+00:00
3438,553959910,2026-05-29 18:00:00+00:00,4,2026-05-27 12:56:00+00:00
